# Rocket League Skillshots: Object-Oriented Pipeline

This notebook rewrites the implementation using an object-oriented structure similar to the FashionMNIST example.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

In [ ]:
LABEL_NAME_MAP = {
    -1: "noise",
    1: "ceiling shot",
    2: "power shot",
    3: "waving dash",
    5: "air dribble",
    6: "front flick",
    7: "musty flick",
}


@dataclass
class PreparedData:
    X_train_t: torch.Tensor
    X_test_t: torch.Tensor
    y_train_t: torch.Tensor
    y_test_t: torch.Tensor
    y_train_np: np.ndarray
    y_test_np: np.ndarray
    index_to_name: dict[int, str]

In [ ]:
class SkillshotDataset:
    """
    Manages parsing, feature engineering, and preprocessing for the
    Rocket League Skillshots dataset.
    """

    def __init__(self, data_path: str | Path, test_size: float = 0.25, random_state: int = 42):
        self.data_path = Path(data_path)
        self.test_size = test_size
        self.random_state = random_state

        self.valid_labels = set(LABEL_NAME_MAP.keys())
        self.binary_cols = [
            "up",
            "accelerate",
            "slow",
            "goal",
            "left",
            "boost",
            "camera",
            "down",
            "right",
            "slide",
            "jump",
        ]

        self.feature_names: list[str] = []
        self.df: pd.DataFrame | None = None
        self.label_to_index: dict[int, int] = {}
        self.index_to_name: dict[int, str] = {}

    def _parse_raw_data(self) -> tuple[list[str], list[pd.DataFrame], list[int]]:
        lines = [
            ln.strip()
            for ln in self.data_path.read_text(encoding="utf-8", errors="replace").splitlines()
            if ln.strip()
        ]
        if not lines:
            raise ValueError("Input file is empty.")

        feature_names = lines[0].split()
        if len(feature_names) != 18:
            raise ValueError(f"Expected 18 feature names in header, got {len(feature_names)}")

        X_sequences: list[pd.DataFrame] = []
        y_labels: list[int] = []

        current_label = None
        current_rows: list[list[float]] = []

        def flush_current_sample() -> None:
            nonlocal current_label, current_rows
            if current_label is None:
                return
            if len(current_rows) == 0:
                seq_df = pd.DataFrame(columns=feature_names)
            else:
                seq_df = pd.DataFrame(current_rows, columns=feature_names)
            X_sequences.append(seq_df)
            y_labels.append(current_label)

        for line_no, line in enumerate(lines[1:], start=2):
            tokens = line.split()

            if len(tokens) == 1:
                try:
                    maybe_label = int(tokens[0])
                except ValueError:
                    maybe_label = None

                if maybe_label in self.valid_labels:
                    flush_current_sample()
                    current_label = maybe_label
                    current_rows = []
                    continue

            if len(tokens) != 18:
                raise ValueError(
                    f"Line {line_no}: expected 18 numeric values or a single label. Got {len(tokens)} token(s)."
                )

            if current_label is None:
                raise ValueError(f"Line {line_no}: found feature row before first label line.")

            try:
                row_values = [float(tok) for tok in tokens]
            except ValueError as exc:
                raise ValueError(f"Line {line_no}: non-numeric value found in feature row.") from exc

            current_rows.append(row_values)

        flush_current_sample()
        return feature_names, X_sequences, y_labels

    def _engineer_sample_features(self, seq_df: pd.DataFrame, feature_names: list[str]) -> dict[str, float]:
        row: dict[str, float] = {}
        active_binary_cols = [c for c in self.binary_cols if c in feature_names]

        for col in feature_names:
            if len(seq_df) == 0:
                row[f"{col}_mean"] = 0.0
                row[f"{col}_std"] = 0.0
                row[f"{col}_min"] = 0.0
                row[f"{col}_max"] = 0.0
            else:
                series = seq_df[col]
                row[f"{col}_mean"] = float(series.mean())
                row[f"{col}_std"] = float(series.std(ddof=0))
                row[f"{col}_min"] = float(series.min())
                row[f"{col}_max"] = float(series.max())

        for col in active_binary_cols:
            row[f"{col}_sum"] = float(seq_df[col].sum()) if len(seq_df) else 0.0

        if "Time" in seq_df.columns and len(seq_df):
            row["duration"] = float(seq_df["Time"].max())
        else:
            row["duration"] = 0.0

        return row

    def build_feature_table(self) -> pd.DataFrame:
        if not self.data_path.exists():
            raise FileNotFoundError(f"Dataset not found at {self.data_path.resolve()}")

        self.feature_names, X_sequences, y_labels = self._parse_raw_data()

        engineered_rows: list[dict[str, float | int | str]] = []
        for seq_df, label in zip(X_sequences, y_labels):
            feat_row = self._engineer_sample_features(seq_df, self.feature_names)
            feat_row["label"] = label
            feat_row["class_name"] = LABEL_NAME_MAP[label]
            engineered_rows.append(feat_row)

        df = pd.DataFrame(engineered_rows).fillna(0)

        sorted_labels = sorted(self.valid_labels)
        self.label_to_index = {lab: idx for idx, lab in enumerate(sorted_labels)}
        index_to_label = {idx: lab for lab, idx in self.label_to_index.items()}
        self.index_to_name = {idx: LABEL_NAME_MAP[lab] for idx, lab in index_to_label.items()}

        df["target"] = df["label"].map(self.label_to_index)
        self.df = df
        return df

    def get_prepared_data(self) -> PreparedData:
        if self.df is None:
            self.build_feature_table()

        assert self.df is not None

        numeric_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        exclude_cols = ["label", "target"]
        numeric_feature_cols = [c for c in numeric_cols if c not in exclude_cols]

        X = self.df[numeric_feature_cols].values
        y = self.df["target"].values

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=self.test_size,
            random_state=self.random_state,
            stratify=y,
        )

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        return PreparedData(
            X_train_t=torch.tensor(X_train_scaled, dtype=torch.float32),
            X_test_t=torch.tensor(X_test_scaled, dtype=torch.float32),
            y_train_t=torch.tensor(y_train, dtype=torch.long),
            y_test_t=torch.tensor(y_test, dtype=torch.long),
            y_train_np=y_train,
            y_test_np=y_test,
            index_to_name=self.index_to_name,
        )

In [ ]:
class LinearClassifier(nn.Module):
    """Simple linear classifier for engineered skillshot features."""

    def __init__(self, input_dim: int, num_classes: int, sigma: float = 0.01):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)
        nn.init.normal_(self.linear.weight, mean=0, std=sigma)
        nn.init.zeros_(self.linear.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x)


class CrossEntropyLossWrapper:
    """Wrapper class for cross-entropy loss."""

    def __init__(self):
        self.criterion = nn.CrossEntropyLoss()

    def __call__(self, predictions: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        return self.criterion(predictions, targets)


class AdamOptimizerWrapper:
    """Wrapper class for Adam optimizer."""

    def __init__(self, model_params, lr: float = 0.01):
        self.optimizer = torch.optim.Adam(model_params, lr=lr)

    def zero_grad(self) -> None:
        self.optimizer.zero_grad()

    def step(self) -> None:
        self.optimizer.step()

In [ ]:
class SkillshotTrainingPipeline:
    """
    Orchestrates training and evaluation using the same OOP pattern
    as the FashionMNIST example.
    """

    def __init__(self, data_path: str | Path, epochs: int = 120, lr: float = 0.01, seed: int = 42):
        self.epochs = epochs
        self.seed = seed
        self._set_seed(seed)

        self.dataset = SkillshotDataset(data_path=data_path, random_state=seed)
        self.prepared_data: PreparedData | None = None

        self.model: LinearClassifier | None = None
        self.loss_fn = CrossEntropyLossWrapper()
        self.optimizer: AdamOptimizerWrapper | None = None

        self.lr = lr
        self.train_losses: list[float] = []
        self.test_accuracies: list[float] = []

    def _set_seed(self, seed: int) -> None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    def setup(self) -> None:
        self.dataset.build_feature_table()
        self.prepared_data = self.dataset.get_prepared_data()

        input_dim = self.prepared_data.X_train_t.shape[1]
        num_classes = len(self.prepared_data.index_to_name)

        self.model = LinearClassifier(input_dim=input_dim, num_classes=num_classes)
        self.optimizer = AdamOptimizerWrapper(self.model.parameters(), lr=self.lr)

    def train(self) -> None:
        if self.prepared_data is None or self.model is None or self.optimizer is None:
            self.setup()

        assert self.prepared_data is not None
        assert self.model is not None
        assert self.optimizer is not None

        X_train_t = self.prepared_data.X_train_t
        y_train_t = self.prepared_data.y_train_t
        X_test_t = self.prepared_data.X_test_t
        y_test_t = self.prepared_data.y_test_t

        self.model.train()

        for epoch in range(self.epochs):
            self.optimizer.zero_grad()

            logits = self.model(X_train_t)
            loss = self.loss_fn(logits, y_train_t)
            loss.backward()
            self.optimizer.step()

            self.train_losses.append(float(loss.item()))

            self.model.eval()
            with torch.no_grad():
                test_logits = self.model(X_test_t)
                y_pred_epoch = torch.argmax(test_logits, dim=1)
                test_acc = (y_pred_epoch == y_test_t).float().mean().item()
            self.test_accuracies.append(float(test_acc))
            self.model.train()

            if (epoch + 1) % 20 == 0 or epoch == 0:
                print(
                    f"Epoch {epoch + 1:03d}/{self.epochs} | "
                    f"Train Loss: {loss.item():.4f} | Test Accuracy: {test_acc:.4f}"
                )

    def evaluate(self) -> dict[str, float | np.ndarray | str]:
        if self.prepared_data is None or self.model is None:
            raise RuntimeError("Call train() before evaluate().")

        self.model.eval()
        with torch.no_grad():
            final_logits = self.model(self.prepared_data.X_test_t)
            y_pred = torch.argmax(final_logits, dim=1).cpu().numpy()

        y_true = self.prepared_data.y_test_np

        accuracy = accuracy_score(y_true, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        )

        ordered_idx = sorted(self.prepared_data.index_to_name.keys())
        ordered_names = [self.prepared_data.index_to_name[i] for i in ordered_idx]

        report = classification_report(y_true, y_pred, target_names=ordered_names, zero_division=0)
        cm = confusion_matrix(y_true, y_pred, labels=ordered_idx)

        return {
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "classification_report": report,
            "confusion_matrix": cm,
            "ordered_names": ordered_names,
        }

    def plot_training_curves(self) -> None:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].plot(self.train_losses)
        axes[0].set_title("Training Loss by Epoch")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Loss")

        axes[1].plot(self.test_accuracies)
        axes[1].set_title("Test Accuracy by Epoch")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Accuracy")

        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_confusion_matrix(cm: np.ndarray, class_names: list[str]) -> None:
        plt.figure(figsize=(9, 7))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
        plt.title("Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.show()

In [ ]:
pipeline = SkillshotTrainingPipeline(
    data_path=Path("data/rocket_league_skillshots.data"),
    epochs=120,
    lr=0.01,
    seed=42,
)

pipeline.train()
metrics = pipeline.evaluate()

print("\nFinal Evaluation Metrics")
print(f"Accuracy : {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall   : {metrics['recall']:.4f}")
print(f"F1 score : {metrics['f1']:.4f}")

print("\nClassification report:")
print(metrics["classification_report"])

pipeline.plot_training_curves()
pipeline.plot_confusion_matrix(metrics["confusion_matrix"], metrics["ordered_names"])